<h3><b><center>Competition Brief: Monthly Lucky Draw Promotion</center></b></h3>

**Overview**

This promotion is designed to reward loyal customers through a monthly lucky draw. Each month, five (5) customers will be randomly selected to win exciting prizes.


**Promotion Period**

The competition will run on a monthly basis. Each qualifying entry within a calendar month will be eligible for that month’s draw.


**Eligibility Criteria**

To qualify for entry into the lucky draw, customers must meet all of the following requirements:
* Purchase at least three (3) products from any participating chocolate brand in a single day
* Achieve a minimum total spend of 30
* Be between the ages of 18 and 35 years

**Entry Mechanism**
* Each qualifying transaction counts as one (1) entry into the monthly draw
* Multiple entries are permitted, provided each transaction meets the qualifying criteria

**Winner Selection**
* At the end of each month, five (5) winners will be selected through a random lucky draw from all eligible entries
* The draw process will be conducted in a fair and auditable manner

**Prizes**
* Winners will receive prizes as determined by the promoter (to be defined per campaign cycle)

**Winner Notification**
* Winners will be contacted using the details provided at the time of purchase or entry
* Failure to respond within a specified timeframe may result in forfeiture and selection of an alternate winner

**General Terms**
* The promoter reserves the right to verify all entries and disqualify any participant who does not meet the eligibility criteria
* Participation in the competition constitutes acceptance of these terms and conditions

**1. Import Packages**

In [1]:
# Data Manipulation Packages
import pandas as pd
import numpy as np
from pandasql import sqldf
from datetime import datetime, timedelta

# Directory Packages
from functools import partial
import os
from pathlib import Path
from dotenv import load_dotenv
import glob

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Package To Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

**2. Create Paths**

In [ ]:
ROOT_DIR: Path = Path().resolve().parent
load_dotenv(os.path.join(ROOT_DIR, ".env"))
Root = os.path.normpath(os.getcwd() + os.sep + os.pardir)
data_path   = Root+"/Data"

**3. Import CSV To Dataframes**

In [27]:
dataframes = {}

for file in glob.glob(os.path.join(data_path, "*.csv")):
    name = os.path.splitext(os.path.basename(file))[0]  # file name without .csv
    dataframes[name] = pd.read_csv(file)

**4. Join All Dataframes**

In [28]:
df = pd.merge(dataframes['sales'], dataframes['products'],on='product_id', how='left')
df = pd.merge(df, dataframes['customers'],on='customer_id', how='left')
df = pd.merge(df, dataframes['stores'],on='store_id', how='left')

**5. Clean Data**

In [29]:
dataframes['sales']

,order_id,order_date,product_id,store_id,customer_id,quantity,unit_price,discount,revenue,cost,profit
0,0RD00000001,2023-01-07,P0080,S093,C040749,5,14.43,0.15,61.33,42.77,18.56
1,0RD00000002,2023-10-22,P0173,S065,C020161,3,12.01,0.00,36.03,19.06,16.97
2,0RD00000003,2023-05-07,P0115,S078,C048069,2,10.02,0.00,20.04,10.29,9.75
3,0RD00000004,2024-06-23,P0186,S088,C047901,2,14.66,0.10,26.39,16.35,10.04
4,0RD00000005,2024-09-24,P0197,S054,C033950,1,12.34,0.00,12.34,7.94,4.40
...,...,...,...,...,...,...,...,...,...,...,...
999995,0RD00999996,2023-04-15,P0103,S053,C044763,4,7.45,0.00,29.80,17.27,12.53
999996,0RD00999997,2023-05-26,P0094,S058,C004441,2,6.09,0.00,12.18,7.81,4.37
999997,0RD00999998,2023-07-05,P0097,S041,C044879,3,6.09,0.00,18.27,10.11,8.16
999998,0RD00999999,2024-01-03,P0090,S083,C042525,1,4.90,0.00,4.90,2.56,2.34


In [6]:
df['order_month'] = df['order_date'].str[:7]
df['order_year'] = df['order_date'].str[:4]

In [7]:
df.head()

,order_id,order_date,product_id,store_id,customer_id,quantity,unit_price,discount,revenue,cost,...,age,gender,loyalty_member,join_date,store_name,city,country,store_type,order_month,order_year
0,0RD00000001,2023-01-07,P0080,S093,C040749,5,14.43,0.15,61.33,42.77,...,44,Male,1,2021-11-17,Chocolate Store 93,Sydney,UK,Airport,2023-01,2023
1,0RD00000002,2023-10-22,P0173,S065,C020161,3,12.01,0.00,36.03,19.06,...,63,Female,1,2023-07-03,Chocolate Store 65,New York,Australia,Retail,2023-10,2023
2,0RD00000003,2023-05-07,P0115,S078,C048069,2,10.02,0.00,20.04,10.29,...,35,Male,1,2023-10-09,Chocolate Store 78,London,UK,Airport,2023-05,2023
3,0RD00000004,2024-06-23,P0186,S088,C047901,2,14.66,0.10,26.39,16.35,...,37,Female,1,2023-05-30,Chocolate Store 88,Toronto,USA,Retail,2024-06,2024
4,0RD00000005,2024-09-24,P0197,S054,C033950,1,12.34,0.00,12.34,7.94,...,57,Female,0,2021-08-20,Chocolate Store 54,London,Canada,Online,2024-09,2024


**6. Query Data Using SQL To Get Qualifying Players**

In [8]:
qualified = """with requirements AS 
                (SELECT customer_id
                        ,order_year
                        ,order_date
                        ,order_month
                        ,quantity
                        ,unit_price
                        ,revenue
                        ,profit
                        ,age
                        ,CASE WHEN quantity >= 3 THEN 1 ELSE 0 END AS MetQuantityReq
                        ,CASE WHEN revenue >= 30 THEN 1 ELSE 0 END AS MetRevenueReq
                        ,CASE WHEN age BETWEEN 18 AND 35 THEN 1 ELSE 0 END AS MetAgeReq
                FROM df
                WHERE loyalty_member = 1
                )
        SELECT * FROM requirements 
        WHERE order_year = '2024'
        AND MetQuantityReq = 1
        AND MetRevenueReq = 1
        AND MetAgeReq = 1
        """

df_qual = sqldf(qualified)

**7. Plot Number Of Qualifying Players**

In [9]:
bar = df_qual.groupby('order_month')['customer_id'].count().reset_index().sort_values(by='order_month')

fig = make_subplots(specs=[[{"secondary_y": True}]])

Legend = dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='right', x=0.6)
x_tick_val = bar['order_month']

fig.add_trace(
    go.Bar(x=bar['order_month'], y=bar['customer_id'], name="Customers",marker=dict(color='blue')),
    secondary_y=False)

fig.update_layout(width=1200,height=600, title_text='<b>Qualifying Players</b>'
                  ,font_family='arial'
                  ,font_color='#202020'
                  ,title_x=0.5
                  ,font_size=15
                  ,plot_bgcolor='white'
                  ,paper_bgcolor='white'
                  ,legend=Legend
                  ,barmode='stack')

fig.update_traces(marker_line_width = 0)
fig.update_yaxes(title_text='<b>Quantity</b>',showgrid=False,range=[0, max(bar['customer_id']) + 10], secondary_y=False)
fig.update_xaxes(title_text='<b>Month </b>', tickvals=x_tick_val, type="category", tickformat='%d-%b-%Y', showgrid=False)

In [10]:
def pick_winners(df, seed=42):
    df = df[['order_month','customer_id','order_date','quantity','revenue','profit','age']]
    X = df.groupby('order_month', group_keys=False).apply(lambda x: x.sample(n=5, random_state=seed)).reset_index(drop=True).sort_values(['order_month'])
    return X

winners = pick_winners(df_qual)

In [11]:
winners.to_csv('Winning Players.csv',index=False)